This notebook tunes the STL order reward for the gate race.

The order reward is the windowed BHNR reward (everything is in `stl.py`): robustness of (¬G_k+1 U G_k) over short overlapping windows, paid when a window closes. The sweep runs through `sweep.py` (parallel, resumable, learning-speed checkpoints) and is compared against plain Q-learning and a label-only control (see Note 4 for what the experiments showed).

In [2]:
# environment check + show() helper.
# This notebook needs matching versions of stl.py, sweep.py and race_env.py next to it.
import importlib
import importlib.util
import inspect
import os

import pandas as pd
from IPython.display import display

REQUIRED = {   # module -> names this notebook uses from it
    "stl": ["STLOrderMonitor", "STL_REWARDS", "robustness_weight", "BHNRWindow"],
    "sweep": ["SETTINGS", "run_jobs", "job_id", "train", "greedy_action", "PLAIN", "LABEL_ONLY", "BASELINE",
              "load_results", "summarize", "compare"],
}
problems = []
for name, attrs in REQUIRED.items():
    try:
        mod = importlib.import_module(name)
    except Exception as e:
        problems.append(f"{name}.py: cannot be imported ({type(e).__name__}: {e})")
        continue
    missing = [a for a in attrs if not hasattr(mod, a)]
    if missing:
        problems.append(f"{name}.py ({os.path.abspath(mod.__file__)}): missing {missing}")
if not problems:
    import stl, sweep
    if "observe_label" not in inspect.signature(stl.STLOrderMonitor.__init__).parameters:
        problems.append("stl.py: STLOrderMonitor has no `observe_label` argument")
    if "course_path" not in inspect.signature(sweep.run_jobs).parameters:
        problems.append("sweep.py: run_jobs has no `course_path` argument")
    if "rho_avoidance" not in sweep.METRICS:
        problems.append("sweep.py: METRICS has no 'rho_avoidance'")
if problems:
    raise RuntimeError("Out-of-date or missing files next to this notebook:\n  - " + "\n  - ".join(problems) +
                       "\nCopy the latest stl.py and sweep.py (plus your race_env.py) into this notebook's folder, then Restart & Run All. "
                       "(bhnr.py, bhnr_sweep.py, make_jobs.py, analyze_sweep.py and hover_analysis.py no longer exist: their code "
                       "is inside stl.py and sweep.py.)")
print("files OK:", ", ".join(f"{m}.py" for m in REQUIRED), "| stl.py =", os.path.abspath(stl.__file__))

# Optional: pandas needs jinja2 for styled tables (and matplotlib for the colour gradients). Without it every table
# below still prints, as plain text.
if importlib.util.find_spec("jinja2") is None:
    print("note: jinja2 is not installed, so tables will print as plain text.  pip install jinja2  for the styled version.")


def show(df, formats=None, precision=2, gradients=()):
    """Display a DataFrame as a styled table (needs jinja2; `gradients` also needs matplotlib), or as plain text if unavailable.
    formats: {column: format string}; gradients: [(column, colormap), ...]"""
    try:
        styler = df.style.format(formats or {}, precision=precision, na_rep="-").hide(axis="index")
        for col, cmap in gradients:
            styler = styler.background_gradient(subset=[col], cmap=cmap)
        display(styler)
    except (ImportError, AttributeError) as e:
        out = df.copy()
        for col in out.columns:
            fmt = (formats or {}).get(col)
            if fmt is not None:
                out[col] = [("-" if pd.isna(v) else fmt.format(v)) for v in out[col]]
            elif out[col].dtype.kind == "f":
                out[col] = [("-" if pd.isna(v) else f"{v:.{precision}f}") for v in out[col]]
        print(f"[plain-text table: {type(e).__name__}: {e}. pip install jinja2 matplotlib for the styled version]")
        print(out.to_string(index=False))


files OK: stl.py, sweep.py | stl.py = /Users/rzhang/Projects/RL_planner/RL Tutorial/Demo Results/stl.py


In [3]:
%load_ext autoreload
%autoreload 2

import os

import numpy as np
import pandas as pd
from IPython.display import display

from stl import robustness_weight
from sweep import (BASELINE, LABEL_ONLY, PLAIN, compare, greedy_action, job_id, load_results, run_jobs, summarize, train)
from sweep import SETTINGS as SWEEP_SETTINGS


In [4]:
COURSE_PATH = "../../ros2_ws/src/pylon_2026_world/purt_course.yaml"
# inherited settings
SETTINGS = {"MAX_STEPS": 400, "NOISE": 0.25, "DISCOUNT_FACTOR": 0.99, "ALPHA": 0.1, "EPSILON": 0.3,
            "test_rollouts": 200, "episodes": 3000}

# parallel training
N_JOBS = max(1, (os.cpu_count() or 2) - 2)

In [5]:
# configs
cell_size = 1.0
first_stage = 1

# ρ_Gk is positive when we stay inside r1 and negative outside
# ρ_¬Gk+1 is positive when we stay outside r2 and negative inside
r1, r2 = 3.0, 9.5
seeds = [67, 123, 42, 7, 99, 2024, 11, 31, 58, 76]  # lap length varies by 3-9 steps across seeds: compare with many seeds

# H_t classes: 0 = NONE (no window closed this step), 1 = NEG (ρ(window) <= 0), 2 = POS (ρ(window) > 0)
h_bins = [0.0]

# BHNR order-reward arms (evidence in Note 4). window = (window_len, overlap) in actions; stride = window_len - overlap.
#   "delta":      reward = w * (ρ(window) - ρ(previous window)). Fastest learning, cannot be farmed by hovering. Best w ~ 3.
#   "robustness": reward = w * ρ(window). Hovering next to the gate earns w * 2.58 / stride per step (2.58 = best
#                 non-crossing hover on this course, from stl.calibrate_hover_rho()), so weights are given as f * stride / 2.58:
#                 f = 1 is break-even, f = 0.5 / 2 / 4 below. Hovering only showed up at f = 8-32, so keep f <= 4.
arms = [
    dict(mode="delta",      window=(2, 1), weights=[1, 3, 10]),
    dict(mode="delta",      window=(3, 2), weights=[3]),
    dict(mode="robustness", window=(1, 0), weights=[round(robustness_weight(f, 1, 0), 4) for f in (0.5, 2, 4)]),
]

# settings handed to sweep.py: its defaults, overridden by this notebook's SETTINGS and the values above
run_settings = {**SWEEP_SETTINGS, **SETTINGS, "CELL_SIZE": cell_size, "FIRST_STAGE": first_stage,
                "R1": r1, "R2": r2, "H_BINS": h_bins}


In [6]:
# Training and evaluation live in sweep.py (train, greedy_action). This cell only keeps the r2-analysis helper below.

def stage_distances(env, Q, n_rollouts):
    rows = []
    for ep in range(n_rollouts):
        state, _ = env.reset()
        d_next = {}
        while True:
            q = env.q
            _, nxt = env.targets()
            state, _, done, truncated, info = env.step(greedy_action(Q, state, env.action_space.n))
            if nxt is not None:
                d_next[q] = min(d_next.get(q, np.inf), env.gate_distance(info["new_xy"], nxt))
            if done or truncated:
                break
        if info["lap_finished"]:
            rows += [{"rollout": ep, "stage": q, "d_next": d} for q, d in d_next.items()]
    return pd.DataFrame(rows)


**Note 1:**
- p_Gk = r1 - d_G_k
- p_Gk+1 = d_G_k+1 - r_2 (positive when outside r2, negative when inside r2)

-> r_2 needs to be > min(d_G_k+1) to make p_Gk+1 (give both negative and postivie rewards)

**Note 2:**
- The distance from the first pylon is only 7.6m, which is smaller than our chosen r_2, which means W will always be negative when we fly from gate 0-1.
- Because the W in from G1-2 and 2-3 is mostly positive, the minimum of W will always be from stage 0 and can never be recovered
-  Reward (old cumulative setup): the order reward was weight × ΔH. In stage 0, ΔH comes from the start geometry, so it's just noise in the reward. A larger weight means more noise, which hurts training.

-> I dropped stage 0 during using our stl order robustness score

**Note 3 (BHNR order reward):**
- Reward and H_t come from ρ(¬G_k+1 U G_k) evaluated only inside short windows (`window_len` actions, `overlap` actions shared with the previous window), with the sup/inf taken over the window, not the whole stage. Windows restart at each gate crossing.
- A closing window pays `w * ρ(window)` (`mode="robustness"`) or `w * (ρ(window) - ρ(previous window))` (`mode="delta"`); H_t is NONE on every other step.
- The `ρ_order` column is still the **full-trajectory** robustness of the order spec (metric only).
- A window's sup reports its best state, so a bad step stays hidden until the best state slides out of the window.
- Stage 0 is excluded (`first_stage = 1`) for the start-geometry reason in Note 2.

**Note 4 (what the experiments showed; 10 seeds per condition unless noted, 3000 episodes, `results.jsonl`):**
- **Use plain Q-learning as the baseline.** `plain` (no STL, no H_t) finishes 100% of laps in 75.5 ± 6.5 steps. The old w = 0 baseline (H_t shown, stage 0 included) is handicapped (90% laps, 86.7 steps; 3 seeds), so comparing against it overstates STL.
- **STL speeds up learning; it does not clearly shorten the final lap.** Greedy lap rate at episode 1000: plain 9%, delta (2,1) w=3 74%, delta (3,2) w=3 70%, robustness (1,0) f=2 60%. Final lap length is not significantly different from plain (best: robustness (1,0) f=0.5, 71.6 ± 2.7 steps, t = -1.75). The effect holds against the label-only control, so it comes from the reward, not from showing H_t.
- **Window shape.** Stride 1 works; stride 2 does not (robustness (3,1): no speedup, 94% laps). Delta is best with windows of 2-3 actions (1 or 6 are worse); robustness is best with the shortest window, and (6,5) gives the longest final laps. At stride 1, `window_len` and `overlap` move together, so they cannot be separated.
- **Weights.** Delta: w ~ 3 is the sweet spot; w = 10 (6 seeds) and 30 still learn fast but end with longer laps (79.0 and 82.1 steps; plain 75.5). Robustness: break-even hovering is f = 1, but in practice (1,0) kept 100% laps and learned faster up to f = 16 (6 seeds each for f = 4-32). It broke at f = 32 (laps 121 ± 47 steps, noise-free greedy rollouts never finish) and f = 8 already had 1 of 6 noise-free failures, so keep f <= 4.
- **Reward size.** The STL term is about 1% of the reward magnitude (up to 7% at delta w = 30); gates and pylon penalties dominate.
- **Compute.** STL adds no measurable training time (about 36-39 s per run for every stride-1 setting, plain included).

In [7]:
# find the min distance to gate k+1 in stages (1, 2)
trained = train(dict(BASELINE, label=False, seed=seeds[0]), course_path=COURSE_PATH, settings=run_settings)   # plain Q-learning
baseline, Q_base = trained.monitor, trained.Q
base_d = stage_distances(baseline, Q_base, SETTINGS["test_rollouts"])

if base_d.empty:
    print(f"The plain baseline finished no laps in {SETTINGS['episodes']} episodes, so the min-distance analysis for r2 "
          "cannot be built. Raise SETTINGS['episodes'] (3000 recommended) and rerun this cell. The sweep below does not depend on it.")
else:
    base_d = base_d[base_d["stage"] >= first_stage] # the baseline run includes stage 0; keep the r2 analysis on the tuned stages

    G = baseline.env.GATES
    n = len(G)
    geometry = []
    for k in range(first_stage, n - 1): 
        nxt = k + 1
        mid = 0.5 * (G[k]["p1"] + G[k]["p2"])
        geometry.append({"stage": k, "Target": G[k]["name"], "Avoid": G[nxt]["name"],
                         "d(mid target, avoid) (m)": baseline.gate_distance(mid, nxt)})
    stage_table = (pd.DataFrame(geometry)
                   .merge(base_d.groupby("stage")["d_next"].agg(["mean", "min"]).reset_index(), on="stage", how="left")
                   .rename(columns={"mean": "Baseline mean min d (m)", "min": "Baseline worst min d (m)"}))
    show(stage_table)

    print(f"Baseline lap rate {base_d['rollout'].nunique() / SETTINGS['test_rollouts']:.0%}; "
          f"smallest baseline min d over all stages = {base_d['d_next'].min():.2f} m")


stage,Target,Avoid,"d(mid target, avoid) (m)",Baseline mean min d (m),Baseline worst min d (m)
1,G2,G3,7.91,10.55,9.07
2,G3,G4,7.86,10.76,7.44


Baseline lap rate 100%; smallest baseline min d over all stages = 7.44 m


In [8]:
# sweep: controls + arms through sweep.py (parallel, resumable, adds learning-speed checkpoints and reward accounting)
# One line per finished run. Runs are keyed by their config only (not SETTINGS / r1 / r2), so delete or rename this file
# if you change those. Rerunning the cell only runs what is missing.
RESULTS_PATH = "results.jsonl"

jobs = []
for seed in seeds:
    jobs += [dict(PLAIN, seed=seed), dict(BASELINE, seed=seed)]       # plain Q-learning; the old w = 0 baseline
    for arm in arms:
        L, ov = arm["window"]
        jobs.append(dict(LABEL_ONLY(L, ov), seed=seed))               # H_t shown, zero reward, same windows
        jobs += [dict(mode=arm["mode"], window_len=L, overlap=ov, w=w, seed=seed) for w in arm["weights"]]
jobs = list({job_id(j): j for j in jobs}.values())                    # drop duplicates

run_jobs(jobs, RESULTS_PATH, workers=N_JOBS, course_path=COURSE_PATH, settings=run_settings)


120 jobs, 0 results already in results.jsonl, 120 to run on 6 worker(s)
[1/120]     35s  delta|L2|ov1|w1|s67|ep3000: lap 100%, steps 76.7 (nf 59), train 30s
[2/120]     35s  delta|L2|ov1|w3|s67|ep3000: lap 100%, steps 76.0 (nf 59), train 31s
[3/120]     38s  robustness|L2|ov1|w0|s67|ep3000|fs1: lap 100%, steps 79.0 (nf 61), train 32s
[4/120]     38s  robustness|L3|ov1|w0|s67|ep3000|fs1|nolabel: lap 100%, steps 80.9 (nf 63), train 32s
[5/120]     39s  delta|L2|ov1|w10|s67|ep3000: lap 100%, steps 85.2 (nf 63), train 34s
[6/120]     46s  robustness|L3|ov1|w0|s67|ep3000: lap 88%, steps 88.7 (nf 71), train 39s
[7/120]     70s  delta|L3|ov2|w3|s67|ep3000: lap 100%, steps 78.5 (nf 61), train 31s
[8/120]     72s  robustness|L1|ov0|w0|s67|ep3000|fs1: lap 100%, steps 69.5 (nf 55), train 30s
[9/120]     72s  robustness|L1|ov0|w0.7752|s67|ep3000: lap 100%, steps 71.3 (nf 55), train 30s
[10/120]     73s  robustness|L3|ov2|w0|s67|ep3000|fs1: lap 100%, steps 84.5 (nf 65), train 32s
[11/120]     75s  

In [9]:
RESULTS_PATH = globals().get("RESULTS_PATH", "results.jsonl")   # set by the sweep cell above
if not os.path.exists(RESULTS_PATH) or os.path.getsize(RESULTS_PATH) == 0:
    print(f"No results in {RESULTS_PATH} yet. Run the sweep cell above first; if it raised an error, fix that and rerun it.")
else:
    runs = load_results(RESULTS_PATH)
    have = sorted(int(e) for e in runs["episodes"].unique())
    runs = runs[runs["episodes"] == SETTINGS["episodes"]]
    if runs.empty:
        print(f"{RESULTS_PATH} has no runs with {SETTINGS['episodes']} episodes (it has: {have}). "
              "Results are only comparable at the same episode count.")
    else:
        results = summarize(runs)
        results = results[results["n_seeds"] >= 3].sort_values(["cond", "mode", "window_len", "overlap", "w"])
        results["window"] = [f"({a},{b})" if c_ == "STL" or c_ == "label-only" else "-" for a, b, c_ in
                             zip(results["window_len"], results["overlap"], results["cond"])]
        table = results[["cond", "mode", "window", "w", "n_seeds", "lap_rate", "lap_steps", "lap_steps_sd", "noisefree_lap_steps",
                         "rho_order", "rho_avoidance", "lap@1000", "auc_1500", "ep_to_90", "train_s", "stl_share"]].rename(columns={
            "cond": "Condition", "mode": "Mode", "window": "Window", "w": "Weight", "n_seeds": "Seeds", "lap_rate": f"Lap @{SETTINGS['episodes']}",
            "lap_steps": "Steps/lap", "lap_steps_sd": "± steps", "noisefree_lap_steps": "Steps (no noise)", "rho_order": "ρ_order (m)",
            "rho_avoidance": "ρ_avoid (m)", "lap@1000": "Lap @1000", "auc_1500": "Learn speed", "ep_to_90": "Eps to 90%",
            "train_s": "Train s", "stl_share": "STL share"})

        print(f"Mean over seeds (rows with >= 3 seeds). Learn speed = mean greedy lap rate at episodes 250-1500; STL share = |STL| / total reward magnitude")
        show(table, formats={f"Lap @{SETTINGS['episodes']}": "{:.1%}", "Lap @1000": "{:.0%}", "Steps/lap": "{:.1f}", "± steps": "{:.1f}",
                         "Steps (no noise)": "{:.1f}", "ρ_order (m)": "{:.2f}", "ρ_avoid (m)": "{:.2f}", "Learn speed": "{:.2f}",
                         "Eps to 90%": "{:.0f}", "Train s": "{:.0f}", "STL share": "{:.1%}", "Weight": "{:.4g}"},
             gradients=[("Steps/lap", "magma_r"), ("Learn speed", "magma")])

        # Welch t per STL arm vs plain Q-learning and vs its matching label-only control (|t| < 2 is roughly seed noise)
        for metric, note in [("lap_steps", "final lap length, lower = better"), ("auc_1500", "learning speed, higher = better")]:
            print(f"\n{metric}: {note}")
            show(compare(runs, metric, min_seeds=5))


Mean over seeds (rows with >= 3 seeds). Learn speed = mean greedy lap rate at episodes 250-1500; STL share = |STL| / total reward magnitude


Condition,Mode,Window,Weight,Seeds,Lap @3000,Steps/lap,± steps,Steps (no noise),ρ_order (m),ρ_avoid (m),Lap @1000,Learn speed,Eps to 90%,Train s,STL share
STL,delta,"(2,1)",1,10,100.0%,77.5,5.9,60.0,1.45,0.45,31%,0.36,1375,32,0.3%
STL,delta,"(2,1)",3,10,100.0%,75.6,3.3,57.6,1.53,0.48,74%,0.49,1150,31,0.8%
STL,delta,"(2,1)",10,10,100.0%,80.1,6.4,61.6,1.47,0.29,57%,0.41,1200,33,2.7%
STL,delta,"(3,2)",3,10,100.0%,77.7,2.7,59.8,1.59,0.40,70%,0.51,1150,32,0.8%
STL,robustness,"(1,0)",0.1938,10,100.0%,71.6,2.7,54.8,1.31,0.56,44%,0.34,1350,31,0.3%
STL,robustness,"(1,0)",0.7752,10,100.0%,73.8,3.5,57.0,1.43,0.52,60%,0.41,1300,30,1.1%
STL,robustness,"(1,0)",1.55,10,100.0%,74.9,5.9,57.6,1.40,0.48,59%,0.37,1425,30,2.0%
baseline(fs0),robustness,-,0,10,94.0%,83.2,6.6,63.0,-2.45,0.36,0%,0.03,2400,40,0.0%
label-only,robustness,"(1,0)",0,10,100.0%,74.7,4.9,57.8,1.26,0.54,10%,0.18,1725,32,0.0%
label-only,robustness,"(2,1)",0,10,99.9%,76.6,3.7,58.6,1.51,0.34,7%,0.18,1675,32,0.0%



lap_steps: final lap length, lower = better


mode,window_len,overlap,w,n,mean,d_vs_plain,t_plain,d_vs_label,t_label
delta,2,1,1.00,10,77.49,2.01,0.72,0.85,0.39
delta,2,1,3.00,10,75.62,0.13,0.06,-1.02,-0.65
delta,2,1,10.00,10,80.09,4.60,1.60,3.45,1.48
delta,3,2,3.00,10,77.66,2.18,0.98,0.22,0.10
robustness,1,0,0.19,10,71.58,-3.90,-1.75,-3.15,-1.77
robustness,1,0,0.78,10,73.84,-1.64,-0.70,-0.89,-0.47
robustness,1,0,1.55,10,74.92,-0.56,-0.20,0.19,0.08



auc_1500: learning speed, higher = better


mode,window_len,overlap,w,n,mean,d_vs_plain,t_plain,d_vs_label,t_label
delta,2,1,1.00,10,0.36,0.24,4.49,0.18,3.42
delta,2,1,3.00,10,0.49,0.37,6.16,0.32,5.22
delta,2,1,10.00,10,0.41,0.29,4.59,0.24,3.71
delta,3,2,3.00,10,0.51,0.38,8.11,0.37,6.98
robustness,1,0,0.19,10,0.34,0.22,3.87,0.16,2.92
robustness,1,0,0.78,10,0.41,0.29,5.02,0.23,4.12
robustness,1,0,1.55,10,0.37,0.25,3.55,0.19,2.78
